# Regression Validation

Loads all city CSVs in `analysis/data/` and checks key metrics against
expected values stored in `analysis/expected_metrics.json`.

Run after any change to `lvt_utils.py`, `viz.py`, or notebook modeling cells
to confirm results haven't changed.

**How to establish a new baseline:** Run all city notebooks, then run
the 'Update expected_metrics.json' cell at the bottom of this notebook.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR    = Path('data')
METRICS_FILE = Path('expected_metrics.json')

# Load all city CSVs
city_dfs = {}
for csv_path in sorted(DATA_DIR.glob('*.csv')):
    if csv_path.name == '.gitkeep':
        continue
    try:
        city_dfs[csv_path.stem] = pd.read_csv(csv_path)
    except Exception as e:
        print(f'ERROR loading {csv_path.name}: {e}')

print(f'Loaded {len(city_dfs)} city CSVs: {sorted(city_dfs.keys())}')

In [ ]:
def compute_metrics(df: pd.DataFrame) -> dict:
    """Compute all comparable metrics for a city dataframe."""
    taxable = df[df['current_tax'] > 0]
    sfr = taxable[taxable['property_category'] == 'Single Family Residential']
    vacant = taxable[taxable['property_category'] == 'Vacant Land']

    metrics = {
        'parcel_count_total':      int(len(df)),
        'parcel_count_taxable':    int(len(taxable)),
        'parcel_count_fully_exempt': int(df['is_fully_exempt'].sum()),
        'total_current_revenue':   float(round(df['current_tax'].sum(), 2)),
        'total_new_revenue':       float(round(df['new_tax'].sum(), 2)),
        'pct_parcels_with_increase': float(round((taxable['tax_change'] > 0).mean() * 100, 2)),
        'pct_parcels_with_decrease': float(round((taxable['tax_change'] < 0).mean() * 100, 2)),
    }

    if len(sfr) > 0:
        metrics['median_tax_change_pct_sfr'] = float(round(sfr['tax_change_pct'].median(), 2))
    else:
        metrics['median_tax_change_pct_sfr'] = None

    if len(vacant) > 0:
        metrics['median_tax_change_pct_vacant'] = float(round(vacant['tax_change_pct'].median(), 2))
    else:
        metrics['median_tax_change_pct_vacant'] = None

    # Income quintile metrics (if Census data present)
    with_income = taxable[taxable['median_income'].notna()]
    if len(with_income) >= 100:
        try:
            with_income = with_income.copy()
            with_income['q'] = pd.qcut(with_income['median_income'], 5, labels=[1,2,3,4,5], duplicates='drop')
            q1 = with_income[with_income['q'] == 1]['tax_change_pct'].median()
            q5 = with_income[with_income['q'] == 5]['tax_change_pct'].median()
            metrics['income_q1_median_tax_change_pct'] = float(round(q1, 2))
            metrics['income_q5_median_tax_change_pct'] = float(round(q5, 2))
        except Exception:
            metrics['income_q1_median_tax_change_pct'] = None
            metrics['income_q5_median_tax_change_pct'] = None
    else:
        metrics['income_q1_median_tax_change_pct'] = None
        metrics['income_q5_median_tax_change_pct'] = None

    return metrics


# Compute current metrics
current_metrics = {city: compute_metrics(df) for city, df in city_dfs.items()}
print('Computed metrics for:', sorted(current_metrics.keys()))

In [ ]:
# Tolerances per metric
TOLERANCES = {
    'parcel_count_total':              {'type': 'exact'},
    'parcel_count_taxable':            {'type': 'exact'},
    'parcel_count_fully_exempt':       {'type': 'exact'},
    'total_current_revenue':           {'type': 'pct', 'tol': 0.5},
    'total_new_revenue':               {'type': 'pct', 'tol': 0.5},
    'pct_parcels_with_increase':       {'type': 'abs', 'tol': 1.0},
    'pct_parcels_with_decrease':       {'type': 'abs', 'tol': 1.0},
    'median_tax_change_pct_sfr':       {'type': 'abs', 'tol': 1.0},
    'median_tax_change_pct_vacant':    {'type': 'abs', 'tol': 2.0},
    'income_q1_median_tax_change_pct': {'type': 'abs', 'tol': 2.0},
    'income_q5_median_tax_change_pct': {'type': 'abs', 'tol': 2.0},
}


def check_metric(name: str, current, expected) -> tuple[str, str]:
    """Returns (PASS/FAIL/SKIP, message)."""
    if expected is None or current is None:
        return 'SKIP', f'null (current={current}, expected={expected})'

    tol_cfg = TOLERANCES.get(name, {'type': 'abs', 'tol': 1.0})

    if tol_cfg['type'] == 'exact':
        if current == expected:
            return 'PASS', f'{current}'
        return 'FAIL', f'{current} ≠ {expected} (exact match required)'

    if tol_cfg['type'] == 'pct':
        diff_pct = abs(current - expected) / abs(expected) * 100 if expected != 0 else 0
        if diff_pct <= tol_cfg['tol']:
            return 'PASS', f'{current:,.2f} (diff {diff_pct:.3f}%)'
        return 'FAIL', f'{current:,.2f} vs {expected:,.2f} ({diff_pct:.2f}% diff, tol {tol_cfg["tol"]}%)'

    if tol_cfg['type'] == 'abs':
        diff = abs(current - expected)
        if diff <= tol_cfg['tol']:
            return 'PASS', f'{current:.2f} (diff {diff:.2f})'
        return 'FAIL', f'{current:.2f} vs {expected:.2f} (diff {diff:.2f}, tol {tol_cfg["tol"]})'

    return 'SKIP', 'unknown tolerance type'


# Load expected metrics (if exists)
if not METRICS_FILE.exists():
    print('No expected_metrics.json found. Run the Update cell below to create one.')
    expected_metrics = {}
else:
    expected_metrics = json.loads(METRICS_FILE.read_text())
    print(f'Loaded expected metrics for: {sorted(expected_metrics.keys())}')

# Run validation
all_passed = True
results_rows = []

for city in sorted(current_metrics.keys()):
    curr = current_metrics[city]
    exp  = expected_metrics.get(city, {})

    if not exp:
        print(f'\n{city.upper()}: NO BASELINE — run Update cell to create one')
        continue

    city_passed = True
    print(f'\n{city.upper()}:')
    for metric in TOLERANCES:
        status, msg = check_metric(metric, curr.get(metric), exp.get(metric))
        icon = '✓' if status == 'PASS' else ('–' if status == 'SKIP' else '✗')
        print(f'  {icon} {metric}: {msg}')
        if status == 'FAIL':
            city_passed = False
            all_passed = False
        results_rows.append({'city': city, 'metric': metric, 'status': status, 'detail': msg})

    print(f'  → {"PASS" if city_passed else "FAIL"}')

print('\n' + '='*60)
print(f'OVERALL: {"ALL PASS" if all_passed else "FAILURES DETECTED — see above"}')

## Update expected_metrics.json

Run this cell after verifying results are correct to save the current metrics as the new baseline.

In [ ]:
# Uncomment and run to save current metrics as new baseline
# METRICS_FILE.write_text(json.dumps(current_metrics, indent=2))
# print(f'Saved {len(current_metrics)} city baselines to {METRICS_FILE}')